## 模块 2：省份 - 品类交叉营收分布统计（多表关联 + 分组聚合）
### 业务需求

基于原始表，统计各省份、各商品品类的营收贡献与订单量，分析区域化品类消费特征

### 思路分析

1. 关联销售订单表、门店信息表、商品信息表 3 张原始表

2. 按省份 + 商品品类分组，聚合统计营收、订单量、营收占比


### 核心知识点

- 多表`INNER JOIN`关联逻辑

- 多维度`GROUP BY`分组聚合

- 占比计算与百分比格式化

- 子查询作为统计分母的使用

### SQL 代码

In [1]:
import pandas as pd
import sqlite3

# 1. 连接SQLite数据库
conn = sqlite3.connect("/kaggle/input/datasets/tclaim/retail-sales/retail_sales.db")

# 4. 封装通用SQL执行函数，自动打印表格结果
def query_sql(sql):
    return pd.read_sql(sql, conn)
    
print("数据导入完成，调用 query_sql('SQL语句') 即可执行查询")

数据导入完成，调用 query_sql('SQL语句') 即可执行查询


In [2]:
sql = '''
SELECT
    s.所在省份,
    p.商品类别,
    SUM(o.销售数量 * o."单价(元)") AS 品类总营收,
    ROUND(SUM(o.销售数量 * o."单价(元)") * 100.0 / (SELECT SUM(销售数量 * "单价(元)") FROM "销售订单表"), 2) AS `营收占比(%)`
FROM "销售订单表" o
INNER JOIN "门店信息表" s ON o.门店ID = s.门店ID
INNER JOIN "商品信息表" p ON o.商品ID = p.商品ID
GROUP BY s.所在省份, p.商品类别
ORDER BY 品类总营收 DESC;
'''
query_sql(sql)

,所在省份,商品类别,品类总营收,营收占比(%)
0,辽宁省,数码家电,970972.0,7.35
1,吉林省,数码家电,842715.0,6.38
2,黑龙江省,数码家电,794440.0,6.01
3,山西省,数码家电,785620.0,5.95
4,西藏自治区,数码家电,628999.0,4.76
...,...,...,...,...
186,安徽省,日用百货,272.0,0.00
187,青海省,饮料酒水,259.0,0.00
188,贵州省,食品零食,184.0,0.00
189,吉林省,日用百货,117.5,0.00


### 结果说明

输出各省份各品类的营收与占比，可直接用于区域化商品运营、供应链备货、区域营销活动制定，精准匹配不同区域的消费偏好